# Eye diagrams — clean vs noisy

Photodetected eye for both regimes, noiseless and with noise, side by side. Shows the dual
nature of the noise: **ASE** beat noise is signal-dependent (upper levels noisier), **thermal**
noise is additive (uniform). Loads the per-regime checkpoints from `results/`. Run from the
`MLforCPO2` folder.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
sys.path.insert(0, "functions")        # local library modules live in functions/

%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import resample_poly

from config import Config
from transmitter import Transmitter, bits_to_symbols
from channel import OpticalChannel
from train import random_bits
from utils import add_awgn, decision_phase_by_separation

%matplotlib inline
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EBN0 = 20.0        # Eb/N0 [dB] for the noisy eye

In [ ]:
def draw_eye(ax, photocurrent, sps, phase, title, fine_sps=40, skip=100, num_symbols=4000):
    """Overlay symbol windows of the photocurrent to form the eye."""
    fine = resample_poly(photocurrent[:(skip + num_symbols + 8) * sps], fine_sps, sps)
    offset = int(round(phase * fine_sps / sps))
    centers = (skip + np.arange(num_symbols)) * fine_sps + offset
    keep = (centers - fine_sps >= 0) & (centers + fine_sps < len(fine))
    centers = centers[keep]
    window = np.arange(-fine_sps, fine_sps)[:, None] + centers[None, :]
    ax.plot(np.arange(-fine_sps, fine_sps) / fine_sps, fine[window], color=(0.85, 0.33, 0.10, 0.04), lw=0.4)
    ax.axvline(0, color="k", linestyle=":", linewidth=1)
    ax.set_xlim(-1, 1)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Symbol time")
    ax.set_ylabel("Photodetected power")
    ax.set_title(title)

In [ ]:
for regime in ["ase", "thermal"]:
    ckpt = torch.load(os.path.join("results", f"pam4_{regime}.pt"), weights_only=False, map_location=device)
    cfg = Config()
    cfg.noise_regime = regime
    cfg.minibatch_symbols = 4096
    cfg.edge_guard_symbols = 48
    cfg.rx_sqrt_companding = ckpt.get("rx_sqrt_companding")                       # restore build choices
    cfg.optical_filter_type = ckpt.get("optical_filter_type", cfg.optical_filter_type)
    tx = Transmitter(cfg).to(device)
    ch = OpticalChannel(cfg).to(device)
    tx.load_state_dict(ckpt["transmitter"])
    tx.eval()
    sps = cfg.samples_per_symbol_sim
    torch.manual_seed(2)
    bits = random_bits(cfg.bits_per_symbol, 6000, device)
    symbols = bits_to_symbols(bits, cfg.bits_per_symbol).cpu().numpy()
    with torch.no_grad():
        clean = ch(tx(bits)).cpu().numpy()
        if regime == "ase":
            noisy = ch(tx(bits), ase_ebn0_db=EBN0).cpu().numpy()                  # ASE beat noise on the field
        else:
            noisy = add_awgn(ch(tx(bits)), EBN0, cfg.bits_per_symbol, sps).cpu().numpy()   # thermal AWGN at ADC
    phase, _ = decision_phase_by_separation(clean, sps, symbols)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    draw_eye(axes[0], clean, sps, phase, f"{regime}: clean (noiseless)")
    draw_eye(axes[1], noisy, sps, phase, f"{regime}: with noise @ Eb/N0 = {EBN0:.0f} dB")
    fig.tight_layout()
    fig.savefig(os.path.join("results", f"pam4_eye_{regime}.png"), dpi=150, bbox_inches="tight")
    plt.show()